# **SCRAPING GOOGLE PLAY STORE (SHOPEE)**

In [8]:
pip install google-play-scraper

In [9]:
# import library utama
from google_play_scraper import reviews, Sort
import pandas as pd
import time
import random

### 1. Script Scraping

In [10]:
# id aplikasi
app_id = 'com.shopee.id'

all_reviews = []
continuation_token = None

TARGET = 30000  # ambil banyak agar fleksibel

while len(all_reviews) < TARGET:
    result, continuation_token = reviews(
        app_id,
        lang='id',
        country='id',
        sort=Sort.NEWEST,
        count=200,
        continuation_token=continuation_token
    )

    for r in result:
        content = r.get('content', '').strip()
        score = r.get('score', None)

        # filter dasar
        if content and score:
            all_reviews.append({
                'review': content,
                'rating': score
            })

    print(f"collected: {len(all_reviews)}")

    if continuation_token is None:
        break

    time.sleep(1)  # hindari rate limit

collected: 200
collected: 400
collected: 600
collected: 800
collected: 1000
collected: 1200
collected: 1400
collected: 1600
collected: 1800
collected: 2000
collected: 2200
collected: 2400
collected: 2600
collected: 2800
collected: 3000
collected: 3200
collected: 3400
collected: 3600
collected: 3800
collected: 4000
collected: 4200
collected: 4400
collected: 4600
collected: 4800
collected: 5000
collected: 5200
collected: 5400
collected: 5600
collected: 5800
collected: 6000
collected: 6200
collected: 6400
collected: 6600
collected: 6800
collected: 7000
collected: 7200
collected: 7400
collected: 7600
collected: 7800
collected: 8000
collected: 8200
collected: 8400
collected: 8600
collected: 8800
collected: 9000
collected: 9200
collected: 9400
collected: 9600
collected: 9800
collected: 10000
collected: 10200
collected: 10400
collected: 10600
collected: 10800
collected: 11000
collected: 11200
collected: 11400
collected: 11600
collected: 11800
collected: 12000
collected: 12200
collected: 12400

### 2. Cleaning Awal + Save Raw Data

In [11]:
# buat dataframe
df = pd.DataFrame(all_reviews)

# hapus duplikasi
df = df.drop_duplicates(subset=['review'])

# buang teks terlalu pendek
df = df[df['review'].str.len() > 5]

# reset index
df = df.reset_index(drop=True)

print("\ndistribusi awal:")
print(df['rating'].value_counts())
print("\njumlah data:", len(df))

# simpan raw
df.to_csv('shopee_reviews_raw.csv', index=False)


distribusi awal:
rating
5    13002
1     6702
2     1121
4     1092
3     1092
Name: count, dtype: int64

jumlah data: 23009


### 3. Labeling + Augmentation + Balancing

In [17]:
# =========================
# labeling + augmentation + balancing
# =========================

# fungsi labeling
def label_sentiment(r):
    if r <= 2:
        return 'negatif'
    elif r == 3:
        return 'netral'
    else:
        return 'positif'

# apply labeling
df['sentiment'] = df['rating'].apply(label_sentiment)

print("distribusi sebelum balancing:")
print(df['sentiment'].value_counts())


# augmentasi sederhana (tukar kata)
def augment_text(text):
    words = text.split()

    # skip jika terlalu pendek
    if len(words) < 4:
        return text

    i, j = random.sample(range(len(words)), 2)
    words[i], words[j] = words[j], words[i]

    return " ".join(words)


# balancing cerdas (real + augment)
def smart_balance(df, label, target_real, target_total):
    subset = df[df['sentiment'] == label]

    # ambil data asli
    if len(subset) >= target_real:
        real_data = subset.sample(n=target_real, random_state=42)
    else:
        real_data = subset.copy()

    augmented = []

    # tambah variasi data
    while len(real_data) + len(augmented) < target_total:
        row = real_data.sample(1).iloc[0]
        new_text = augment_text(row['review'])

        augmented.append({
            'review': new_text,
            'rating': row['rating'],
            'sentiment': row['sentiment']
        })

    df_aug = pd.DataFrame(augmented)

    return pd.concat([real_data, df_aug])


# target balancing (tidak terlalu agresif)
TARGET_REAL = 3000
TARGET_TOTAL = 4000

# apply balancing
df_neg = smart_balance(df, 'negatif', TARGET_REAL, TARGET_TOTAL)
df_net = smart_balance(df, 'netral', 1000, TARGET_TOTAL)  # netral data sedikit
df_pos = smart_balance(df, 'positif', TARGET_REAL, TARGET_TOTAL)

# gabungkan semua
df_balanced = pd.concat([df_neg, df_net, df_pos])

# shuffle data
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\ndistribusi setelah balancing:")
print(df_balanced['sentiment'].value_counts())

print("\ntotal data:", len(df_balanced))

distribusi sebelum balancing:
sentiment
positif    14094
negatif     7823
netral      1092
Name: count, dtype: int64

distribusi setelah balancing:
sentiment
negatif    4000
netral     4000
positif    4000
Name: count, dtype: int64

total data: 12000


### 4. Save Dataset

In [18]:
# simpan dataset final (yang akan dipakai training)
df_balanced.to_csv('shopee_reviews_final.csv', index=False)